# Índice Oceánico de El Niño (ONI) – Serie mensual con rezagos

Convierte el archivo `raw_data/oni.ascii.txt` (NOAA CPC) en una serie mensual

`processed_data/oni_mensual.csv`

**Formato original:** el ONI es la anomalía de la temperatura superficial del mar en la región Niño 3.4,
promediada en **trimestres móviles** (`DJF` = diciembre-enero-febrero, `JFM` = enero-febrero-marzo, …).
La columna `YR` corresponde al año del **mes central** del trimestre (por eso `NDJ 1950` = nov-1950, dic-1950, ene-1951).

**Asignación al último mes (para evitar filtración de información futura):**
si el trimestre se asignara a su mes central, el valor de enero (DJF) incluiría datos de febrero, que en enero todavía
no se conocen (*data leakage*). Por eso cada trimestre se asigna a su **último mes**, cuando el dato ya está completo: `DJF` → febrero, `JFM` → marzo, …, `NDJ` → enero del año siguiente.

**Rezagos:** se agregan `ONI_lag1`, `ONI_lag3`, `ONI_lag6` y `ONI_lag12` (ONI de t−1, t−3, t−6 y t−12) para pronosticar mas adelante el precio en t.
Además de evitar la filtración, y tambien porque se espera que el efecto de El Niño / La Niña sobre cosechas y pastos
tarda varios meses en llegar a los precios (corto plazo para el arroz, largo plazo para la ganadería).

en futuros analisis se usaran los rezagos dado que se publica el valor de un trimestre a inicios del mes siguiente (p. ej. `JAS` se publica en octubre), así que en el mes t aún no se conoce el `ONI` asignado a t. La columna `ONI` (sin rezago) sirve para análisis descriptivo, no como predictor.

## 1. Preparar el entorno

In [ ]:
import os
import sys

REPO_URL = "https://github.com/yevasa93/AnomaliasClimaticasArrozCarne.git"
REPO_DIR = "AnomaliasClimaticasArrozCarne"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # En Colab se clona el repositorio público (o se actualiza si ya existe)
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !git -C {REPO_DIR} pull
    os.chdir(REPO_DIR)

print("Directorio de trabajo:", os.getcwd())

In [ ]:
import pandas as pd

RAW_FILE = os.path.join("raw_data", "oni.ascii.txt")
OUT_DIR = "processed_data"
OUT_FILE = os.path.join(OUT_DIR, "oni_mensual.csv")

os.makedirs(OUT_DIR, exist_ok=True)

## 2. Lectura del archivo original

El archivo separa las columnas con espacios, por eso se lee con `sep=r"\s+"` (no hace falta convertirlo a CSV antes).

In [ ]:
oni_raw = pd.read_csv(RAW_FILE, sep=r"\s+")
print(oni_raw.shape)
oni_raw.head(13)

## 3. Asignación de cada trimestre a su último mes

| Trimestre | Último mes | Año |
|---|---|---|
| DJF | 2 (febrero) | YR |
| JFM | 3 (marzo) | YR |
| … | … | … |
| OND | 12 (diciembre) | YR |
| NDJ | 1 (enero) | **YR + 1** |

In [ ]:
ULTIMO_MES = {
    "DJF": 2, "JFM": 3, "FMA": 4, "MAM": 5, "AMJ": 6, "MJJ": 7,
    "JJA": 8, "JAS": 9, "ASO": 10, "SON": 11, "OND": 12, "NDJ": 1,
}

oni = oni_raw.rename(columns={"SEAS": "Trimestre", "TOTAL": "SST_Nino34", "ANOM": "ONI"})

desconocidos = set(oni["Trimestre"]) - set(ULTIMO_MES)
assert not desconocidos, f"Trimestres no reconocidos: {desconocidos}"

oni["Mes"] = oni["Trimestre"].map(ULTIMO_MES)
# NDJ termina en enero del año siguiente
oni["Anio"] = oni["YR"] + (oni["Trimestre"] == "NDJ").astype(int)
oni["Fecha"] = pd.to_datetime(dict(year=oni["Anio"], month=oni["Mes"], day=1))

oni = oni.sort_values("Fecha").reset_index(drop=True)
oni[["Trimestre", "YR", "Fecha", "Anio", "Mes", "SST_Nino34", "ONI"]].iloc[8:16]

## 4. Validaciones

In [ ]:
print("Rango:", oni["Fecha"].min().date(), "a", oni["Fecha"].max().date())

# Debe haber exactamente un valor por mes, sin huecos (requisito para que los rezagos sean correctos)
assert oni["Fecha"].is_unique, "Hay meses duplicados"
meses_esperados = pd.date_range(oni["Fecha"].min(), oni["Fecha"].max(), freq="MS")
faltantes = meses_esperados.difference(oni["Fecha"])
assert len(faltantes) == 0, f"Meses faltantes: {list(faltantes)}"
assert oni["ONI"].notna().all()

print(f"{len(oni)} meses continuos, sin duplicados ni faltantes")

## 5. Rezagos y fase ENSO

- `ONI_lag1`, `ONI_lag3`, `ONI_lag6`, `ONI_lag12`: valor del ONI 1, 3, 6 y 12 meses antes. Se calculan sobre la serie
  mensual completa (antes de cruzar con SIPSA) para que los rezagos de 2013 usen datos de 2012.
- `Fase_ENSO`: clasificación mensual según el umbral de ±0.5 °C (`El Niño` ≥ 0.5, `La Niña` ≤ −0.5, `Neutral` en otro caso).
  Es una clasificación simplificada: la definición oficial de un *episodio* exige que el umbral se mantenga
  al menos 5 trimestres consecutivos.

In [ ]:
REZAGOS = [1, 3, 6, 12]

for k in REZAGOS:
    oni[f"ONI_lag{k}"] = oni["ONI"].shift(k)

oni["Fase_ENSO"] = pd.cut(
    oni["ONI"],
    bins=[-float("inf"), -0.5, 0.5 - 1e-9, float("inf")],
    labels=["La Niña", "Neutral", "El Niño"],
).astype(str)

columnas = ["Fecha", "Anio", "Mes", "Trimestre", "SST_Nino34", "ONI", "Fase_ENSO"] + [f"ONI_lag{k}" for k in REZAGOS]
oni = oni[columnas]
oni.tail(13)

In [ ]:
# Distribución de fases en el periodo de la serie SIPSA (2013 en adelante)
oni[oni["Anio"] >= 2013].groupby("Anio")["Fase_ENSO"].value_counts().unstack(fill_value=0)

## 6. Exportar CSV

In [ ]:
oni.to_csv(OUT_FILE, index=False, encoding="utf-8", date_format="%Y-%m-%d")
print(f"Archivo generado: {OUT_FILE} ({len(oni)} filas, {os.path.getsize(OUT_FILE) / 1024:.0f} KB)")

In [ ]:
# En Colab: descargar el CSV generado para subirlo al repositorio
if EN_COLAB:
    from google.colab import files
    files.download(OUT_FILE)

## 7. Lectura del CSV desde el repositorio público de GitHub

In [ ]:
URL_ONI = (
    "https://raw.githubusercontent.com/yevasa93/AnomaliasClimaticasArrozCarne/main/"
    "processed_data/oni_mensual.csv"
)

oni = pd.read_csv(URL_ONI, parse_dates=["Fecha"])
print(oni.shape)
oni.tail()